# Feature engineering — Base 3 — PM2.5 em Pequim

Orquestra `features/temporal.py` com `BASE_3`.

Passo de `1h`. Horizonte final pendente.

## Disponibilidade

- Poluentes e meteorologia: simultâneos ao alvo → estado de `t` para prever `t+1`.
- `No`, `station` e `wd` bruto: **proibidos** (`wd_sin`/`wd_cos` já preparados).
- Sem interpolação; linhas incompletas saem do quadro.
- Arquivos: `catalogo_features_base3.csv` e `disponibilidade_covariaveis_base3.csv`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.features.regras_bases import BASE_3, validar_proibidas
from series_temporais.features.temporal import (
    build_one_step_frame,
    choque_futuro_nao_altera_features,
    chronological_train_test,
)

BASE_DIR = ROOT / "data" / "base_03"
DATA_PATH = BASE_DIR / "prepared.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base3.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base3.csv"
SPEC = BASE_3
TRAIN_RATIO = 0.8


## Quadro de um passo

Cada linha é uma origem. O alvo do modelo é a variação até o passo seguinte; ele não entra nas features.

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(SPEC.time_col, kind="stable").reset_index(drop=True)
validar_proibidas(SPEC.external_cols(), SPEC)
quadro, features = build_one_step_frame(
    df,
    time_col=SPEC.time_col,
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert len(quadro) == 14231
assert len(features) == 29
assert {"y_true", "target_delta", "target_time"}.isdisjoint(features)
quadro[features].head()


,level_t,PM10_t,SO2_t,NO2_t,CO_t,O3_t,TEMP_t,PRES_t,DEWP_t,RAIN_t,...,target_mean_24,target_std_24,target_mean_168,target_std_168,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_month_sin,target_month_cos
0,344.0,389.0,107.0,195.0,3799.0,84.0,4.1,998.2,-1.3,0.0,...,263.375000,43.503652,110.428571,100.339161,0.258819,9.659258e-01,-0.433884,-0.900969,0.866025,0.5
1,339.0,370.0,90.0,177.0,4099.0,84.0,3.3,998.5,-1.7,0.0,...,267.291667,46.266870,112.452381,101.593132,0.500000,8.660254e-01,-0.433884,-0.900969,0.866025,0.5
2,315.0,346.0,47.0,144.0,3200.0,84.0,1.4,999.0,-1.5,0.0,...,282.166667,49.963042,120.190476,105.499490,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5
3,310.0,327.0,45.0,146.0,3299.0,84.0,1.1,999.7,-1.8,0.0,...,283.833333,50.361747,122.035714,106.177462,0.965926,-2.588190e-01,-0.433884,-0.900969,0.866025,0.5
4,309.0,318.0,44.0,145.0,4099.0,84.0,5.7,1000.4,0.5,0.0,...,284.000000,50.441385,123.863095,106.753327,0.866025,-5.000000e-01,-0.433884,-0.900969,0.866025,0.5


## Catálogo

O arquivo CSV fica na pasta do grupo e registra, para cada coluna de `X`, se a informação é conhecida na origem, só no passado ou antecipadamente.

In [3]:
catalogo = SPEC.catalogo(features)
disponibilidade = SPEC.tabela_disponibilidade()
assert not catalogo.grupo.eq("revisar").any()
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
disponibilidade


,coluna,disponibilidade,atraso,feature,status,justificativa
0,PM10,na_origem,0,PM10_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
1,SO2,na_origem,0,SO2_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
2,NO2,na_origem,0,NO2_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
3,CO,na_origem,0,CO_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
4,O3,na_origem,0,O3_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
5,TEMP,na_origem,0,TEMP_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
6,PRES,na_origem,0,PRES_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
7,DEWP,na_origem,0,DEWP_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
8,RAIN,na_origem,0,RAIN_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...
9,WSPM,na_origem,0,WSPM_t,em_validacao,Simultânea ao alvo; usada só como estado de t ...


## Corte cronológico

A última linha de treino cujo alvo cruzaria a primeira origem de teste é removida.

In [4]:
treino, teste = chronological_train_test(quadro, TRAIN_RATIO)
assert len(treino) == 11383
assert len(teste) == 2847
assert treino.target_time.max() < teste.origin_time.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.origin_time.min(), teste.origin_time.min()],
    "ultima_origem": [treino.origin_time.max(), teste.origin_time.max()],
})


,recorte,linhas,primeira_origem,ultima_origem
0,treino,11383,2013-03-08 00:00:00,2016-05-28 13:00:00
1,teste,2847,2016-05-28 15:00:00,2017-02-20 10:00:00


## Checagem de leakage

A segunda metade da série recebe um valor artificial. As features das origens anteriores precisam permanecer idênticas.

In [5]:
resultado = choque_futuro_nao_altera_features(
    df,
    time_col=SPEC.time_col,
    indice_inicial_do_choque=len(df) // 2,
    colunas=[SPEC.target_col, *SPEC.external_cols()],
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert resultado["iguais"] is True
assert resultado["diferenca_maxima"] == 0
resultado


{'origens_comparadas': 8661, 'diferenca_maxima': 0.0, 'iguais': True}